# Trajectory Analysis

One of the applications of SC RNA-Seq is the ability to understand how cells evolve in their transcriptional landscape over time, basically this is a trajectory of the transcription over time.

Trajectory analysis tries to order single cells along a continuous path that represents a biological process such as differentiation, activation, or response to stimulus. It does not give absolute time. Instead it gives pseudotime, a relative ordering that puts “earlier” cells at one end and “later” cells at the other.

Use trajectory analysis when:
- You expect a continuous process (development, differentiation, activation).
- Cells are sampled from different stages of that process, not just random unrelated cell types.
- Do not use it when the population is a mix of distinct unrelated cell types with no transitional states.

In our case study, we are working with Fetal Liver hematopoietic stem cell that has the potential to differentiate into different types of blood cells

### Install Packages!

In [ ]:
!pip install scanpy
!pip install anndata
!pip3 install igraph
!pip install celltypist
!pip install decoupler
!pip install fa2-modified
!pip install louvain
!pip install scvelo

### Load Data

In [ ]:
#Import core single cell datasets

import scanpy as sc
import anndata as ad
import numpy as np
#import scvelo as scv

In [ ]:
!mkdir -p GSM5115832
!wget https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5115nnn/GSM5115832/suppl/GSM5115832%5FE11%5F5%5FmAGM%5FFL%5Fbarcodes.tsv.gz -O /content/GSM5115832/barcodes.tsv.gz
!wget https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5115nnn/GSM5115832/suppl/GSM5115832%5FE11%5F5%5FmAGM%5FFL%5Ffeatures.tsv.gz -O /content/GSM5115832/features.tsv.gz
!wget https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5115nnn/GSM5115832/suppl/GSM5115832%5FE11%5F5%5FmAGM%5FFL%5Fmatrix.mtx.gz -O /content/GSM5115832/matrix.mtx.gz

🚧 Note that this is another way to import data obtained from 10x.

In [ ]:
esc_adata = sc.read_10x_mtx('/content/GSM5115832/')
esc_adata.var_names_make_unique()

### QC

In [ ]:
esc_adata.shape

In [ ]:
esc_adata.var.head()

In [ ]:
esc_adata.var['MT'] = esc_adata.var_names.str.startswith("MT-")
esc_adata.var['RIBO'] = esc_adata.var_names.str.startswith("RPS", "RPL")
esc_adata.var['HB'] = esc_adata.var_names.str.startswith("^HB[^(P)]")

In [ ]:
sc.pp.calculate_qc_metrics(
    esc_adata, qc_vars=["MT", 'RIBO', 'HB'], inplace=True, log1p=True
)

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (5,4)  # Adjust figure size
plt.rcParams["axes.grid"] = True  # Add grid to plots
plt.rcParams["axes.edgecolor"] = "black" # Set plot border color
plt.rcParams["axes.linewidth"] = 1.5 # Set plot border width
plt.rcParams["axes.facecolor"] = "white" # Set background color
plt.rcParams["axes.labelcolor"] = "black" # Set label color
plt.rcParams["xtick.color"] = "black" # Set x-axis tick color
plt.rcParams["ytick.color"] = "black" # Set y-axis tick color
plt.rcParams["text.color"] = "black" # Set text color

In [ ]:
sc.pl.violin(
    esc_adata,
    ["n_genes_by_counts", 'total_counts', 'pct_counts_MT'],
    jitter=0.4,
    multi_panel=False,
)

In [ ]:
sc.pl.scatter(esc_adata, "total_counts", "n_genes_by_counts", color="pct_counts_MT")

In [ ]:
#sc.pp.scrublet(esc_adata)

In [ ]:
#Normalisation
esc_adata.layers["counts"] = esc_adata.X.copy()
sc.pp.normalize_total(esc_adata)
sc.pp.log1p(esc_adata)

In [ ]:
#Feature selection
sc.pp.highly_variable_genes(esc_adata, n_top_genes=2500)
sc.pl.highly_variable_genes(esc_adata)

### Dimensionality Reduction

In [ ]:
#Dim Reduction
sc.tl.pca(esc_adata)
sc.pl.pca_variance_ratio(esc_adata, n_pcs=10, log=False)


In [ ]:
il_genes = [gene for gene in esc_adata.var_names[esc_adata.var['highly_variable']] if gene.lower().startswith('il')]
print(il_genes)

In [ ]:
sc.pl.pca(esc_adata, color="Il12a", cmap="coolwarm")

### UMAP

In [ ]:
sc.pp.neighbors(esc_adata)
sc.tl.umap(esc_adata)

In [ ]:
esc_adata

In [ ]:
sc.pl.umap(
    esc_adata,
    color=["Cdh23"],
    size=8,
)

In [ ]:
sc.tl.leiden(esc_adata, flavor="igraph", n_iterations=2, key_added="leiden_res0_5", resolution=0.25)

In [ ]:
sc.pl.umap(
    esc_adata,
    color=["leiden_res0_5"],
    size=32,
)

### Cell Type Annotation

In [ ]:
import decoupler as dc

In [ ]:
# Query Omnipath and get PanglaoDB
markers = dc.op.resource(name="PanglaoDB", organism="mouse")

# Keep canonical cell type markers alone
markers = markers[markers["mouse"]]

# Remove duplicated entries
markers = markers[~markers.duplicated(["cell_type", "genesymbol"])]

# Format because dc only accepts cell_type and genesymbol

markers = markers.rename(columns={"cell_type": "source", "genesymbol": "target"})
markers = markers[["source", "target"]]


markers.head()

In [ ]:
esc_adata.var_names

In [ ]:
dc.mt.ulm(data=esc_adata,
          net=markers,
          tmin = 3)

In [ ]:
score = dc.pp.get_obsm(esc_adata, key="score_ulm")

In [ ]:
esc_adata.obsm["score_ulm"].head(1)

In [ ]:
esc_adata.obsm["score_ulm"].columns

In [ ]:
#rank genes
esc_adata_gene_rank = dc.tl.rankby_group(score, groupby="leiden_res0_5", reference="rest", method="t-test_overestim_var")
esc_adata_gene_rank = esc_adata_gene_rank[esc_adata_gene_rank["stat"] > 0]
esc_adata_gene_rank.head(5)

In [ ]:
top_cell_type_per_group = esc_adata_gene_rank.groupby('group')['name'].apply(lambda x: x.head(1))
display(top_cell_type_per_group.to_dict())

In [ ]:
sc.pl.umap(score, color=["Hepatoblasts","leiden_res0_5"], cmap="RdBu_r")

In [ ]:
dict_ann = esc_adata_gene_rank[esc_adata_gene_rank["stat"] > 0].groupby("group").head(1).set_index("group")["name"].to_dict()
dict_ann

In [ ]:
esc_adata.obs["leiden_res0_5"] = esc_adata.obs["leiden_res0_5"].cat.rename_categories(dict_ann)

In [ ]:
sc.pl.umap(
    adata=esc_adata,
    color=[ "leiden_res0_5"],
    ncols=1,
)

### Trajectory Inference
It is an attempt to understand how cells transition from one type to another (like stem → mature)?

#### First, we will build a graph?

Think of a **graph** as a network:

- Each **cell** is a **dot (node)**.
- If two cells are similar, draw a **line (edge)** between them.

So the graph connects cells that look alike.

In [ ]:
#Trajectory analysis
sc.tl.draw_graph(esc_adata)

In [ ]:
plt.rcParams["figure.figsize"] = (4,4)
sc.pl.draw_graph(esc_adata, color='leiden_res0_5', size = 16)

#### Then we will ABSTRACT the graph

Basically, all the points that cluster to make one cell type, will be converted to one point. More like a blunt summary of everypoint

In [ ]:
sc.tl.paga(esc_adata, groups='leiden_res0_5')


In [ ]:
sc.pl.paga(esc_adata, color=['leiden_res0_5'])

In [ ]:
sc.tl.draw_graph(esc_adata, init_pos='paga')

In [ ]:
sc.pl.draw_graph(esc_adata, color='leiden_res0_5', legend_loc='on data', size=8)

In [ ]:
plt.rcParams["figure.figsize"] = (5,4)
sc.pl.paga_compare(esc_adata, threshold=0.03, frameon=True, edges=True, size = 16)

#### Now how do cells transition from one type to another, assuming we have a pluripotent progenitor cell.

In [ ]:
esc_adata.uns['iroot'] = np.flatnonzero(esc_adata.obs['leiden_res0_5']  == 'Pluripotent stem cells')[0]
sc.tl.dpt(esc_adata)

In [ ]:
sc.pl.draw_graph(esc_adata, color=['dpt_pseudotime', 'leiden_res0_5'], legend_loc='on data', size = 24)

In [ ]:
esc_adata.write("esc_adata.h5", compression="gzip")